In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent

In [99]:
loader = PyPDFLoader("../data/1.pdf")
docs = loader.load()

In [100]:
len(docs)

11

In [101]:
spilitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = spilitter.split_documents(docs)

In [102]:
len(splitted_docs)

43

In [103]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = InMemoryVectorStore.from_documents(
    documents = splitted_docs, 
    embedding = embeddings
    )

In [104]:
same_record = vectorstore.similarity_search("Education")

In [105]:
same_record[0].page_content

'connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\n2'

### TOOLS LLM PROMPTS

In [106]:
@tool
def retriever_tool(query:str):
    """Retrieves relevant information from the vectorstore based on the query."""
    print(f"Query received: {query}")
    same_record = vectorstore.similarity_search(query)
    context = ""
    for record in same_record:
        context += record.page_content + "\n"
    return context

In [107]:
print(retriever_tool.invoke("projects"))

Query received: projects
transduction problems such as language modeling and machine translation [ 29, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [31, 21, 13].
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the ﬁrst Transformer models and
has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head
attention and the parameter-free position representation and became the other person involved in nearly every
detail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and
tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and
efﬁcient inference and visualizations. Lukasz and Aidan spent countless long days design

In [108]:
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [109]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="You are a helpful assistant. ALWAYS call retriever_tool first to search the documents before answering any question and if you want you can call retriever_tool multiple times before answering. Answer only from the retrieved context."
)

In [113]:
query = "Do two seprate search for Model Variations and Multi-Head Attention"
response = agent.invoke({"messages":[{"role": "user", "content": query}]})

Query received: Model Variations
Query received: Multi-Head Attention


In [111]:
result = response["messages"][-1].content

In [112]:
print(result)

The Encoder and Decoder stacks are fundamental components in sequence-to-sequence models, such as the Transformer model. These stacks are used to process input sequences and generate output sequences, respectively.

### Encoder Stack
The Encoder stack is composed of a series of identical layers, each containing two sub-layers. The first sub-layer is a multi-head self-attention mechanism, which allows the encoder to focus on different parts of the input sequence. The second sub-layer is a position-wise, fully connected feed-forward network. This stack maps an input sequence of symbol representations (x1,...,x n) to a sequence of continuous representations z = (z1,...,z n).

### Decoder Stack
The Decoder stack is also composed of a series of identical layers, each containing three sub-layers. In addition to the two sub-layers found in the encoder, the decoder includes a third sub-layer that performs multi-head attention over the output of the encoder stack. This allows the decoder to inc